# Explainer Notebook: Jutland Migration, Gentrification, and District Change in Copenhagen

## 1. Motivation

This project studies a recurring Copenhagen narrative: *"Jutlanders are taking over the city."*

We use district-level data to ask a stricter question:

**Where do Jutland-born residents settle over time, and how does that pattern align with district-level education profiles often associated with gentrification?**

The website is built for a non-technical audience and leads with narrative + visual evidence.  
This notebook documents the reproducible pipeline, data cleaning choices, assumptions, and analytical limits.

### Why these datasets?

- `residents.xlsx` (KKBEF9) gives long-run internal migration structure by district and birth region.
- `education_attainment_dataset.csv` (cleaned KKUDD2 export) provides district-level education composition through 2024.

Together, they allow district-year panel analysis linking settlement patterns and socioeconomic profile shifts.

### End-user experience goal

Build a story where readers can:
- inspect spatial distribution (map),
- inspect temporal dynamics (timeline),
- test district-level patterns themselves (explorer),
- and understand uncertainty/limitations without reading technical details first.


## 2. Dataset and Provenance

Canonical input files in this repo:

1. **`residents.xlsx`** (KKBEF9): district, sex, age, birth-region counts, 1977-2026.  
2. **`education_attainment_dataset.csv`** (cleaned KKUDD2 export): district-level education composition, 1985-2024.

Provenance notes:

- `merge-csv.com__69f9cb92c6f8b.csv` is the raw ISO-8859 intermediate export used in earlier cleaning.
- `district_year_panel.csv` is the derived merged panel used by site assets.
- `analysis_summary.json`, `web_metrics.json`, and `plots/*.png` are pipeline outputs from `build_story_assets.py`.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path('.')

summary = json.loads((ROOT / 'analysis_summary.json').read_text(encoding='utf-8'))
web_metrics = json.loads((ROOT / 'web_metrics.json').read_text(encoding='utf-8'))

print('Loaded summary keys:', sorted(summary.keys()))
print('Web metrics years:', web_metrics['years'][0], 'to', web_metrics['years'][-1])


## 3. Basic Stats and Preprocessing

We replicate the cleaning + merge logic used for the story assets, then verify consistency with the committed panel.

Cleaning choices:

- auto-detect year row in residents spreadsheet,
- forward-fill merged dimension cells,
- keep only `Alder i alt` (age-total) rows,
- harmonize district naming (`Vesterbro/Kongens Enghave` -> `Vesterbro-Kongens Enghave`),
- aggregate men + women,
- filter education to `Age total`, `Sex total`, district rows,
- compute `high_ed_share_pct` from
  - `Vocational bachelors educations and bachelors programs`
  - `Masters and PhD programs`.


In [ ]:
def load_residents(path: Path):
    raw = pd.read_excel(path, sheet_name=0, header=None)
    year_count_by_row = raw.apply(
        lambda r: pd.to_numeric(r.iloc[4:], errors='coerce').between(1900, 2100).sum(),
        axis=1,
    )
    year_row_idx = int(year_count_by_row.idxmax())
    years = [
        int(v)
        for v in pd.to_numeric(raw.iloc[year_row_idx, 4:], errors='coerce').dropna().tolist()
    ]

    cols = ['age_group', 'sex', 'neighborhood_raw', 'birth_region'] + years
    df = raw.iloc[year_row_idx + 1:, :len(cols)].copy()
    df.columns = cols

    for c in ['age_group', 'sex', 'neighborhood_raw', 'birth_region']:
        df[c] = df[c].ffill()

    df = df[df['age_group'] == 'Alder i alt'].copy()
    for y in years:
        df[y] = pd.to_numeric(df[y], errors='coerce').fillna(0)

    name_map = {'Vesterbro/Kongens Enghave': 'Vesterbro-Kongens Enghave'}
    df['district'] = (
        df['neighborhood_raw'].astype(str)
        .str.replace('Bydel - ', '', regex=False)
        .replace(name_map)
    )
    return df, years


def reshape_residents(df: pd.DataFrame, years):
    long_df = df.melt(
        id_vars=['district', 'birth_region', 'sex'],
        value_vars=years,
        var_name='year',
        value_name='count',
    )
    long_df['year'] = long_df['year'].astype(int)
    long_df['count'] = pd.to_numeric(long_df['count'], errors='coerce').fillna(0)
    long_df = (
        long_df.groupby(['district', 'birth_region', 'year'], as_index=False)['count']
        .sum()
        .sort_values(['district', 'birth_region', 'year'])
    )
    return long_df


def load_education(path: Path):
    ed = pd.read_csv(path)
    ed['year'] = pd.to_numeric(ed['year'], errors='coerce')
    ed['value'] = pd.to_numeric(ed['value'], errors='coerce')

    ed = ed[
        (ed['age_group'] == 'Age total')
        & (ed['sex'] == 'Sex total')
        & (ed['area'].str.startswith('District - ', na=False))
    ].copy()

    ed['district'] = (
        ed['area']
        .str.replace('District - ', '', regex=False)
        .str.replace('Vesterbro/Kongens Enghave', 'Vesterbro-Kongens Enghave', regex=False)
    )
    return ed


def build_panel(res_long: pd.DataFrame, ed: pd.DataFrame):
    res_total = (
        res_long.groupby(['district', 'year'], as_index=False)['count']
        .sum()
        .rename(columns={'count': 'total_residents'})
    )

    jutland_regions = ['Nordjylland', 'Vestjylland', 'Østjylland', 'Sydjylland']
    local_region = 'Københavns Kommune'

    jut = (
        res_long[res_long['birth_region'].isin(jutland_regions)]
        .groupby(['district', 'year'], as_index=False)['count']
        .sum()
        .rename(columns={'count': 'jutland_residents'})
    )
    local = (
        res_long[res_long['birth_region'] == local_region]
        .groupby(['district', 'year'], as_index=False)['count']
        .sum()
        .rename(columns={'count': 'local_residents'})
    )

    residents_panel = res_total.merge(jut, on=['district', 'year'], how='left').merge(
        local, on=['district', 'year'], how='left'
    )
    residents_panel[['jutland_residents', 'local_residents']] = residents_panel[
        ['jutland_residents', 'local_residents']
    ].fillna(0)

    residents_panel['jutland_share_pct'] = 100 * residents_panel['jutland_residents'] / residents_panel['total_residents']
    residents_panel['local_share_pct'] = 100 * residents_panel['local_residents'] / residents_panel['total_residents']

    high_ed_levels = {
        'Vocational bachelors educations and bachelors programs',
        'Masters and PhD programs',
    }

    total_ed = (
        ed[ed['education_level'] == 'Highest education completed total']
        .groupby(['district', 'year'], as_index=False)['value']
        .sum()
        .rename(columns={'value': 'edu_total'})
    )
    high_ed = (
        ed[ed['education_level'].isin(high_ed_levels)]
        .groupby(['district', 'year'], as_index=False)['value']
        .sum()
        .rename(columns={'value': 'edu_high'})
    )

    edu_panel = total_ed.merge(high_ed, on=['district', 'year'], how='left')
    edu_panel['edu_high'] = edu_panel['edu_high'].fillna(0)
    edu_panel['high_ed_share_pct'] = 100 * edu_panel['edu_high'] / edu_panel['edu_total']

    residents_panel = residents_panel[
        ~residents_panel['district'].str.contains('Uden for inddeling', case=False, na=False)
    ].copy()

    panel = residents_panel.merge(edu_panel, on=['district', 'year'], how='inner')
    panel = panel.replace([np.inf, -np.inf], np.nan).dropna(
        subset=['jutland_share_pct', 'local_share_pct', 'high_ed_share_pct']
    )

    city = (
        residents_panel.groupby('year', as_index=False)[
            ['jutland_residents', 'local_residents', 'total_residents']
        ]
        .sum()
        .sort_values('year')
    )
    city['jutland_share_pct'] = 100 * city['jutland_residents'] / city['total_residents']
    city['local_share_pct'] = 100 * city['local_residents'] / city['total_residents']

    return panel, city


In [ ]:
residents_raw, residents_years = load_residents(ROOT / 'residents.xlsx')
residents_long = reshape_residents(residents_raw, residents_years)
education = load_education(ROOT / 'education_attainment_dataset.csv')

panel_rebuilt, city_rebuilt = build_panel(residents_long, education)
panel_saved = pd.read_csv(ROOT / 'district_year_panel.csv')

print('Residents years:', min(residents_years), 'to', max(residents_years))
print('Rebuilt panel shape:', panel_rebuilt.shape)
print('Saved panel shape:', panel_saved.shape)
print('District count:', panel_rebuilt['district'].nunique())
print('Panel year range:', int(panel_rebuilt['year'].min()), '-', int(panel_rebuilt['year'].max()))


In [ ]:
compare_cols = [
    'total_residents', 'jutland_residents', 'local_residents',
    'jutland_share_pct', 'local_share_pct', 'edu_total', 'edu_high', 'high_ed_share_pct'
]

merged = panel_saved.merge(
    panel_rebuilt,
    on=['district', 'year'],
    suffixes=('_saved', '_rebuilt'),
    how='inner'
)

max_diffs = {}
for col in compare_cols:
    diff = (merged[f'{col}_saved'] - merged[f'{col}_rebuilt']).abs().max()
    max_diffs[col] = float(diff)

print('Max absolute diffs saved vs rebuilt panel:')
for k, v in max_diffs.items():
    print(f'  {k}: {v:.10f}')

city_1985 = city_rebuilt[city_rebuilt['year'] == 1985].iloc[0]
city_2024 = city_rebuilt[city_rebuilt['year'] == 2024].iloc[0]

print('\nCitywide shares (rebuilt):')
print('  Jutland share 1985 -> 2024:', round(city_1985['jutland_share_pct'], 2), '->', round(city_2024['jutland_share_pct'], 2))
print('  Local share   1985 -> 2024:', round(city_1985['local_share_pct'], 2), '->', round(city_2024['local_share_pct'], 2))

latest = panel_rebuilt[panel_rebuilt['year'] == 2024].sort_values('jutland_share_pct', ascending=False)
print('\nTop 3 districts by Jutland share in 2024:')
print(latest[['district', 'jutland_share_pct', 'high_ed_share_pct']].head(3).round(2).to_string(index=False))


### Key points from EDA

- Citywide Jutland-born share changes moderately across the panel period.  
- Citywide København-born share falls more strongly.  
- Top Jutland-share districts in 2024 are central/transformed districts (Vesterbro-Kongens Enghave, Indre By, Nørrebro).  
- District-level education share and Jutland share are positively associated in the merged panel.


### Course Methods Applied

This project combines methods practiced across the lecture notebooks:

- **Week 2 (data wrangling and schema alignment):** cleaned messy table exports, harmonized district naming, and built merge-ready tables.
- **Week 3-4 (comparative and relationship analysis):** compared districts over time and used correlation-oriented visual reasoning.
- **Week 5 (geospatial visualization):** built district choropleth interaction with neighborhood polygons.
- **Week 6 (explanatory interactivity):** combined overview plots with filterable user controls and details-on-demand.
- **Week 7 (web storytelling):** delivered the analysis as a structured one-page narrative website.
- **Week 8 (Segel & Heer narrative design):** used martini-glass structure and explicit visual/narrative tools.


## 4. Data Analysis

We measure association between district high-education share and district Jutland-born share across district-years.

This is an **associative** analysis:
- no causal identification strategy,
- no individual-level migration pathways,
- and no subgroup education disaggregation for Jutland-born vs København-born individuals in this export.


In [ ]:
corr_all = panel_rebuilt['jutland_share_pct'].corr(panel_rebuilt['high_ed_share_pct'])
corr_2024 = panel_rebuilt[panel_rebuilt['year'] == 2024]['jutland_share_pct'].corr(
    panel_rebuilt[panel_rebuilt['year'] == 2024]['high_ed_share_pct']
)

print('Correlation (all district-years):', round(float(corr_all), 3))
print('Correlation (year 2024):', round(float(corr_2024), 3))

base = panel_rebuilt[panel_rebuilt['year'] == 1985][['district', 'jutland_share_pct', 'high_ed_share_pct']].set_index('district')
end = panel_rebuilt[panel_rebuilt['year'] == 2024][['district', 'jutland_share_pct', 'high_ed_share_pct']].set_index('district')
change = end.join(base, lsuffix='_2024', rsuffix='_1985')
change['jutland_share_change'] = change['jutland_share_pct_2024'] - change['jutland_share_pct_1985']
change['high_ed_share_change'] = change['high_ed_share_pct_2024'] - change['high_ed_share_pct_1985']

print('\nLargest increase in Jutland share (1985 -> 2024):')
print(change.sort_values('jutland_share_change', ascending=False).head(5).round(2).to_string())


## 5. Genre and Narrative Design (Segel & Heer)

### Story genre

We use a **martini-glass** structure:
- guided narrative sequence for core claims,
- followed by interactive tools for reader-driven inspection.

### Visual Narrative tools (Figure 7)

- **Visual Structuring**: website is divided into ordered panels (hook -> map -> timeline -> evidence -> limits).  
- **Highlighting**: key finding cards and focused district rankings direct attention to important patterns.  
- **Transition Guidance**: story shifts from spatial patterning to temporal change to socioeconomic association.

### Narrative Structure tools (Figure 7)

- **Ordering**: macro-to-micro progression and explicit year-scope notes.  
- **Interactivity**: map controls, timeline hover, and district explorer controls.  
- **Messaging**: explicit framing of uncertainty and non-causal interpretation.


## 6. Visualization Choices

- **Interactive choropleth map** (`map.html`): best for spatial heterogeneity and filter-based exploration.  
- **Interactive multi-line timeline** (`time_plot.html`): best for long-run district trajectories.  
- **Citywide origin trend plot**: compact macro context.  
- **District ranking bar chart**: direct latest-year comparison.  
- **Education-vs-Jutland scatter with fit line**: relationship view for panel-wide association.  
- **Paired trend panel for top-growth districts**: temporal co-movement evidence.

These complement each other: interaction for exploration, static plots for explanation.


In [ ]:
# Optional full regeneration of site assets from canonical inputs.
# Set to True when you want to refresh all derived files before final hand-in.

import subprocess
import sys

RUN_ASSET_PIPELINE = False

if RUN_ASSET_PIPELINE:
    subprocess.run([sys.executable, 'build_story_assets.py'], check=True)
    print('Asset pipeline completed.')
else:
    print('Pipeline execution skipped (set RUN_ASSET_PIPELINE=True to regenerate assets).')


In [ ]:
from pathlib import Path

plot_paths = [
    'plots/citywide_origin_shares.png',
    'plots/district_jutland_share_latest.png',
    'plots/education_vs_jutland_scatter.png',
    'plots/top_growth_districts_trends.png',
]

try:
    from IPython.display import Image, display
    for path in plot_paths:
        display(Image(filename=path))
except Exception:
    for path in plot_paths:
        exists = Path(path).exists()
        print(path, 'exists=', exists)


## 7. Discussion

### What went well

- Long time horizon allows robust temporal storytelling.
- Joining origin and education profiles produces a stronger, more interpretable narrative than origin-only mapping.
- The one-page website supports both guided reading and audience exploration.

### What is still missing / what could improve

- Education export currently limits subgroup-level comparison by origin.
- No causal model (e.g., policy shock, quasi-experimental variation) is used.
- Additional housing-price and rent-series integration could strengthen the gentrification interpretation.


## 8. Contributions

Replace placeholders with real names before submission:

- **Member A (placeholder):** data cleaning pipeline, merged panel construction, reproducibility scripts.
- **Member B (placeholder):** website implementation, interactive map/timeline/explorer integration.
- **Member C (placeholder):** narrative design, interpretation, and explainer notebook write-up.

Do not submit with "all contributed equally".


## 9. References

1. København Statistikbank, **KKBEF9**: population by district, sex, age, and place of birth.  
2. København Statistikbank, **KKUDD2**: educational attainment by ancestry, age, sex, education, and district.  
3. Segel, E., and Heer, J. (2010). *Narrative Visualization: Telling Stories with Data*.
